In [ ]:
from langchain.embeddings import HuggingFaceInstructEmbeddings
from langchain.document_loaders import DirectoryLoader
from langchain.embeddings import HuggingFaceHubEmbeddings,HuggingFaceInstructEmbeddings
from langchain.llms import HuggingFaceHub
from langchain.vectorstores import Chroma
from langchain.text_splitter import CharacterTextSplitter
from langchain.chains.question_answering import load_qa_chain
from langchain.chains import VectorDBQA
from langchain.llms import OpenAI
from langchain.llms.base import LLM
import torch
import time
from datasets import load_dataset
from typing import List
from huggingface_hub import InferenceClient

In [ ]:
import pandas as pd
from tqdm import tqdm
import time
import os

In [ ]:
pd.options.display.max_rows=5000
pd.options.display.max_colwidth=3000

In [ ]:
import replicate
# os.environ['REPLICATE_API_TOKEN']='' 
os.environ['REPLICATE_API_TOKEN']='' 

In [ ]:
PROMPT='''
You are an Indian legal expert, having the knowledge of constitution of india, acts, statues etc. You will be provided with an objective question along with options. Your task is to generate context that can be used to answer the question correctly.

Question: {question}

Options: {option}

Context:

NOTE:1. Generate a context based on the question and the option to answer the question correctly.
     2. DO NOT PROVIDE DIRECT ANSWER TO THE QUESTION IN THE CONTEXT
'''

In [ ]:
dataset = load_dataset("opennyaiorg/aibe_dataset")
dataset=dataset['train'].to_pandas()

In [ ]:
rows=[]
for i in tqdm(range(len(dataset))):
    exam_name=dataset['exam_name'][i]
    exam_number=dataset['exam_number'][i]
    question_number=dataset['question_number'][i]
    question_text=dataset['question_text'][i]
    opt=dataset['options'][i]
    options=f"A) {opt['A']} \n B) {opt['B']} \n C) {opt['C']} \n D) {opt['D']}"
    input_text=PROMPT.format(question=question_text,option=options)
    input={
                "top_k": 50,
                "top_p": 0.9,
                "prompt": input_text,
                "max_tokens": 4096,
                "min_tokens": 0,
                "temperature": 0.6,
                "prompt_template": "<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n\nYou are a helpful assistant<|eot_id|><|start_header_id|>user<|end_header_id|>\n\n{prompt}<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n\n",
                "presence_penalty": 1.15,
                "frequency_penalty": 0.2
            }
    
    for k in range(2):
        try:

            output=replicate.run(
                        "meta/meta-llama-3-70b-instruct",
                        input=input
                    )
            generated_text=""
            for item in output:
                generated_text+=item+''
            answer=dataset['answer'][i]
            rows.append([exam_number,question_number,question_text,options,generated_text,answer])
            break
        except Exception as e:
            print('Retrying=>',k+1)
            print(e)
            pass
            time.sleep(3)


In [ ]:
result_df=pd.DataFrame(rows,columns=['exam_number','question_number','question_text','options','context','answer'])

In [ ]:
dataset.to_csv('',index=False)

In [ ]:
# len(rows)